# Task 4 — Retail Data Integration

**Topic:** Combine Excel, SQL, and Python  
**Dataset:** Indian FMCG Retail Sales Customer Inventory (2024)

### Workflow
1. Load and clean the retail dataset using Python.
2. Save the cleaned CSV.
3. Perform SQL analysis in MySQL (documented separately in `Retail_Data_Integration_Task_4.sql`).
4. Create the Excel analysis workbook using Python/Pandas.


## 1. Import Libraries and Load Raw Dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Indian FMCG Retail Sales Customer Inventory (2024).csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 100000
Columns: 21


,Invoice_ID,Invoice_Date,City,Store_Format,Category,Brand,Channel,Payment_Mode,Units,Cost_Price,...,Revenue,Cost,Margin,Margin_%,Stock_On_Hand,Reorder_Level,Lead_Time_Days,Customer_Age,Customer_Gender,Loyalty_Flag
0,58018430,2/2/2024 13:50,Kolkata,Super,Grocery,Nestle,Online,Wallet,2,148.999035,...,350.114418,297.998070,52.116348,0.148855,154,31,11,20.0,M,1
1,48157952,10/9/2024 11:52,Hyderabad,Hyper,Home Care,PepsiCo,Offline,Wallet,1,80.626759,...,106.034197,80.626759,25.407438,0.239616,130,24,5,26.0,F,1
2,23283831,8/26/2024 22:03,Chennai,Hyper,Snacks,PepsiCo,Online,Card,5,181.233170,...,1044.900114,906.165848,138.734266,0.132773,270,44,8,27.0,F,0
3,53537460,6/9/2024 4:34,Bengaluru,Hyper,Beverages,Nestle,Omnichannel,Card,2,195.811402,...,521.759920,391.622805,130.137115,0.249420,403,63,8,NaN,F,0
4,55348596,6/7/2024 1:13,Delhi,Super,Snacks,ITC,Offline,Card,3,23.571888,...,91.873416,70.715665,21.157751,0.230292,366,40,9,60.0,F,1


## 2. Check Missing Values

In [2]:
df.isnull().sum()

Invoice_ID             0
Invoice_Date           0
City                   0
Store_Format           0
Category               0
Brand                  0
Channel                0
Payment_Mode           0
Units                  0
Cost_Price             0
Selling_Price          0
Revenue                0
Cost                   0
Margin                 0
Margin_%               0
Stock_On_Hand          0
Reorder_Level          0
Lead_Time_Days         0
Customer_Age       40081
Customer_Gender     5048
Loyalty_Flag           0
dtype: int64

In [3]:
missing_value = df.isnull().sum()
missing_value[missing_value > 0]

Customer_Age       40081
Customer_Gender     5048
dtype: int64

## 3. Check Duplicate Rows

In [4]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


## 4. Inspect Dataset Structure

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 21 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Invoice_ID       100000 non-null  int64  
 1   Invoice_Date     100000 non-null  str    
 2   City             100000 non-null  str    
 3   Store_Format     100000 non-null  str    
 4   Category         100000 non-null  str    
 5   Brand            100000 non-null  str    
 6   Channel          100000 non-null  str    
 7   Payment_Mode     100000 non-null  str    
 8   Units            100000 non-null  int64  
 9   Cost_Price       100000 non-null  float64
 10  Selling_Price    100000 non-null  float64
 11  Revenue          100000 non-null  float64
 12  Cost             100000 non-null  float64
 13  Margin           100000 non-null  float64
 14  Margin_%         100000 non-null  float64
 15  Stock_On_Hand    100000 non-null  int64  
 16  Reorder_Level    100000 non-null  int64  
 17  Lea

## 5. Convert Invoice Date to Datetime

In [6]:
df["Invoice_Date"] = pd.to_datetime(
    df["Invoice_Date"],
    errors="coerce"
)

print(df["Invoice_Date"].dtype)

datetime64[us]


## 6. Handle Missing Customer Age

In [7]:
df["Customer_Age"] = df["Customer_Age"].fillna(
    df["Customer_Age"].median()
)

print(
    "Missing Customer_Age:",
    df["Customer_Age"].isnull().sum()
)

Missing Customer_Age: 0


## 7. Handle Missing Customer Gender

In [8]:
df["Customer_Gender"] = df["Customer_Gender"].fillna(
    df["Customer_Gender"].mode()[0]
)

print(
    "Missing Customer_Gender:",
    df["Customer_Gender"].isnull().sum()
)

Missing Customer_Gender: 0


## 8. Verify Missing Values After Cleaning

In [9]:
print("Total missing values:", df.isnull().sum().sum())

Total missing values: 0


## 9. Check Invalid Numeric Values

In [10]:
print("Units <= 0:", (df["Units"] <= 0).sum())
print("Cost Price <= 0:", (df["Cost_Price"] <= 0).sum())
print("Selling Price <= 0:", (df["Selling_Price"] <= 0).sum())
print("Revenue <= 0:", (df["Revenue"] <= 0).sum())

Units <= 0: 0
Cost Price <= 0: 0
Selling Price <= 0: 0
Revenue <= 0: 0


In [11]:
print("Margin_% below 0:", (df["Margin_%"] < 0).sum())
print("Margin_% above 100:", (df["Margin_%"] > 100).sum())

Margin_% below 0: 0
Margin_% above 100: 0


## 10. Final Cleaning Verification

In [12]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Total missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Rows: 100000
Columns: 21
Total missing values: 0
Duplicate rows: 0


## 11. Before vs After Comparison

In [13]:
comparison = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Missing Values",
        "Duplicate Rows"
    ],
    "Before": [
        100000,
        21,
        45129,
        0
    ],
    "After": [
        df.shape[0],
        df.shape[1],
        df.isnull().sum().sum(),
        df.duplicated().sum()
    ]
})

comparison

,Metric,Before,After
0,Rows,100000,100000
1,Columns,21,21
2,Missing Values,45129,0
3,Duplicate Rows,0,0


## 12. Save Cleaned CSV

In [14]:
df.to_csv("cleaned_retail_data.csv", index=False)

print("Cleaned CSV created successfully!")

Cleaned CSV created successfully!


## 13. SQL Analysis

The cleaned dataset was imported into MySQL and analyzed using SQL.

The SQL queries are saved separately in:

`Retail_Data_Integration_Task_4.sql`

The main analyses include:
- Total revenue
- Total cost and margin
- Revenue by category
- Revenue by city
- Revenue by sales channel
- Revenue by brand
- Average order value
- Revenue by payment mode
- Revenue by store format
- Monthly revenue
- Category and channel revenue


## 14. Load Cleaned Data for Excel Analysis

In [15]:
df = pd.read_csv("cleaned_retail_data.csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 100000
Columns: 21


## 15. Summary Metrics

In [16]:
summary = pd.DataFrame({
    "Metric": [
        "Total Revenue",
        "Total Cost",
        "Total Margin",
        "Average Order Value"
    ],
    "Value": [
        round(df["Revenue"].sum(), 2),
        round(df["Cost"].sum(), 2),
        round(df["Margin"].sum(), 2),
        round(df["Revenue"].sum() / df["Invoice_ID"].nunique(), 2)
    ]
})

summary

,Metric,Value
0,Total Revenue,39335134.98
1,Total Cost,31461449.33
2,Total Margin,7873685.66
3,Average Order Value,393.60


## 16. Revenue by Category

In [17]:
category = (
    df.groupby("Category", as_index=False)["Revenue"]
    .sum()
    .round(2)
    .sort_values("Revenue", ascending=False)
)

category

,Category,Revenue
2,Fruits,4985417.90
3,Grocery,4947363.17
6,Snacks,4942554.66
0,Beverages,4941030.17
4,Home Care,4931420.48
7,Vegetables,4915301.88
5,Personal Care,4884187.45
1,Dairy,4787859.29


## 17. Revenue by City

In [18]:
city = (
    df.groupby("City", as_index=False)["Revenue"]
    .sum()
    .round(2)
    .sort_values("Revenue", ascending=False)
)

city

,City,Revenue
5,Kolkata,4999465.15
3,Delhi,4985262.21
4,Hyderabad,4950786.97
2,Chennai,4900659.09
7,Pune,4899356.64
6,Mumbai,4897600.18
0,Ahmedabad,4877774.67
1,Bengaluru,4824230.07


## 18. Revenue by Channel

In [19]:
channel = (
    df.groupby("Channel", as_index=False)["Revenue"]
    .sum()
    .round(2)
    .sort_values("Revenue", ascending=False)
)

channel

,Channel,Revenue
1,Omnichannel,13162992.40
2,Online,13117659.01
0,Offline,13054483.57


## 19. Revenue by Brand

In [20]:
brand = (
    df.groupby("Brand", as_index=False)["Revenue"]
    .sum()
    .round(2)
    .sort_values("Revenue", ascending=False)
)

brand

,Brand,Revenue
3,ITC,5019512.90
2,HUL,4967523.37
1,Britannia,4942030.33
4,Nestle,4941904.04
0,Amul,4937874.07
7,Tata,4868135.99
6,PepsiCo,4853369.56
5,Parle,4804784.72


## 20. Monthly Revenue

In [21]:
df["Invoice_Date"] = pd.to_datetime(
    df["Invoice_Date"],
    errors="coerce"
)

monthly = (
    df.groupby(df["Invoice_Date"].dt.to_period("M"))["Revenue"]
    .sum()
    .reset_index()
)

monthly["Invoice_Date"] = monthly["Invoice_Date"].astype(str)
monthly = monthly.rename(columns={"Invoice_Date": "Month"})
monthly["Revenue"] = monthly["Revenue"].round(2)

monthly

,Month,Revenue
0,2024-01,3365380.56
1,2024-02,3140896.09
2,2024-03,3350248.06
3,2024-04,3280835.98
4,2024-05,3346824.18
5,2024-06,3175117.85
6,2024-07,3346063.43
7,2024-08,3375015.06
8,2024-09,3211407.61
9,2024-10,3330184.25


## 21. Revenue by Store Format

In [22]:
store = (
    df.groupby("Store_Format", as_index=False)["Revenue"]
    .sum()
    .round(2)
    .sort_values("Revenue", ascending=False)
)

store

,Store_Format,Revenue
1,Hyper,13225626.24
2,Super,13099594.73
0,Express,13009914.01


## 22. Revenue by Category and Channel

In [23]:
category_channel = (
    df.groupby(["Category", "Channel"], as_index=False)["Revenue"]
    .sum()
    .round(2)
    .sort_values(
        ["Category", "Revenue"],
        ascending=[True, False]
    )
)

category_channel

,Category,Channel,Revenue
0,Beverages,Offline,1668306.33
2,Beverages,Online,1663970.80
1,Beverages,Omnichannel,1608753.04
5,Dairy,Online,1619667.46
4,Dairy,Omnichannel,1608152.18
3,Dairy,Offline,1560039.65
6,Fruits,Offline,1687698.97
7,Fruits,Omnichannel,1650724.50
8,Fruits,Online,1646994.42
10,Grocery,Omnichannel,1691965.02


## 23. Create Final Excel Workbook

In [26]:
# Install openpyxl once if it is not already installed:
!pip install openpyxl

with pd.ExcelWriter(
    "Retail_Data_Analysis.xlsx",
    engine="openpyxl"
) as writer:
    summary.to_excel(writer, sheet_name="Summary", index=False)
    category.to_excel(
        writer,
        sheet_name="Revenue by Category",
        index=False
    )
    city.to_excel(
        writer,
        sheet_name="Revenue by City",
        index=False
    )
    channel.to_excel(
        writer,
        sheet_name="Revenue by Channel",
        index=False
    )
    brand.to_excel(
        writer,
        sheet_name="Revenue by Brand",
        index=False
    )
    monthly.to_excel(
        writer,
        sheet_name="Monthly Revenue",
        index=False
    )
    store.to_excel(
        writer,
        sheet_name="Store Format",
        index=False
    )
    category_channel.to_excel(
        writer,
        sheet_name="Category & Channel",
        index=False
    )

print("Excel file created successfully!")

Excel file created successfully!



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 24. Verify Excel File

In [ ]:
import os

print(
    "Retail_Data_Analysis.xlsx exists:",
    os.path.exists("Retail_Data_Analysis.xlsx")
) 

Retail_Data_Analysis.xlsx exists: True
